# Reproducing Thompson & Gold (2026) from dandiset 002034

**Paper.** Thompson LW, Gold JI (2026). Limitations of pupil diameter as a proxy for modes of locus
coeruleus activity. eLife reviewed preprint 112525, version 1 (2 October 2026),
[doi:10.7554/elife.112525](https://doi.org/10.7554/elife.112525). The claims file was written from
the full text of that version as served by the publisher (main text, figure legends, Methods and
the legends of Supplemental Figures 1 and 2).

**Dandiset.** [DANDI:002034](https://dandiarchive.org/dandiset/002034/0.261019.2146), version
0.261019.2146. The paper cites no version; its code's README names this one, and it is the only
published version. The draft holds the same 83 assets with the same total size; it is not used.

**What the dandiset holds.** 83 sessions (32 for monkey Oz, 51 for monkey Cicero, "Ci" in the
paper), one NWB file each, with the trial table, the pupil trace of every trial and 105 sorted LC
units. The paper reports 42 units from 32 sessions (Oz) and 63 units from 51 sessions (Ci).

**Outside inputs.** From the paper's code (https://github.com/lwthompson2/Thompson2026_LCPupilAnalysis,
commit cf3c0b7): its include lists of sessions and units, compared with the dandiset in the link
claim, and its printed statistics on this dandiset (`Code/Results/manuscript_stats_nwb.txt`), used
only as reference values in the link claim. No outside data enters any other claim.

**Readings.** Every method detail is the claims file's, which cites the Methods and the code; the
helper's docstring says where each parameter comes from. The deciding readings are the Methods' trial
rule and windows with the evoked pupil read as E1 (largest window mean minus baseline) or E2
(largest in absolute size, sign kept). Variants: the code's trial rule and windows with its own
evoked pupil (`code`), and two mixtures that isolate the evoked pupil window (`code trials + E1`,
`Methods trials + code pupil`).

No random numbers are drawn anywhere in this notebook.

| Claim | Status |
|---|---|
| include-list-link (link, added by this attempt) | tested |
| data-units-sessions | tested |
| fig2-row1-baseline-lc-pupil | tested |
| fig2-row2-evoked-lc-pupil | tested |
| fig2-row3-pupil-evoked-vs-baseline | tested |
| fig2-row4-lc-evoked-vs-baseline | tested |
| fig2-row5-baseline-lc-vs-evoked-pupil | tested |
| fig2-row6-baseline-pupil-vs-evoked-lc | tested |
| cross-epoch-no-reliable-relation | tested |
| range-baseline-pupil-evoked-lc | tested |
| range-baseline-lc-evoked-pupil | tested |
| figs1-raw-baselines-similar | tested |
| figs2-llr-pupil-baseline-evoked | tested |
| figs2-llr-lc-baseline-evoked | tested |
| figs2-llr-cross-epoch | tested |
| fig3-pooled-within-epoch | tested |
| fig3-pooled-cross-epoch-null | tested |

In [ ]:
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import lcpupil as L

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
T0 = time.time()
print("dandiset", L.DANDISET_ID, "version", L.DANDISET_VERSION)
meta = L.version_metadata()
print(meta["name"])
print(meta["description"])
print([r["url"] for r in meta.get("relatedResource", [])])
assets = L.list_assets()
sessions = L.load_all(assets)
print(f"{len(sessions)} files streamed in {time.time() - T0:.0f} s")

## Data check

The cell below prints what was loaded and checks the description against the values: counts of
subjects, sessions, units and trials; the trial table's columns; the pupil series (units, rate,
whether the Methods' preprocessing is already in it); trial durations and beep times against the
Methods; spike times (sorted, how they are anchored); duplicated recordings; and whether spikes and
pupil share the trial table's clock.

In [ ]:
inv = pd.DataFrame([dict(monkey=s["monkey"], session=s["session"], path=p, n_trials=len(s["trials"]),
                         n_correct=int((s["trials"].correct == 1).sum()),
                         n_beep=int(np.isfinite(s["trials"].beep_time_ms).sum()),
                         n_units=len(s["units"]), samples_per_trial=s["pupil"].shape[1],
                         columns=",".join(s["trial_columns"]))
                    for s, p in zip(sessions, assets.path)])
print("files per subject folder:", assets.path.str.split("/").str[0].value_counts().to_dict())
print(inv.groupby("monkey")[["n_trials", "n_correct", "n_beep", "n_units"]].agg(["count", "sum", "min", "max"]))
print("subject_id matches folder:", all(p.split("/")[0] == "sub-" + m for p, m in zip(inv.path, inv.monkey)))
print("trial table columns in every file:", inv["columns"].unique())
print("pupil series attributes:", {str(s["pupil_attrs"]) for s in sessions})
print("samples per trial (padded with NaN):", inv.samples_per_trial.min(), "to", inv.samples_per_trial.max())
print("spike time policies:", {u["policy"] for s in sessions for u in s["units"]})
print("trial anchors at 100 s spacing in every file:",
      all(np.allclose(s["trials"].start_time.values, 100 * np.arange(len(s["trials"]))) for s in sessions))
print("spike times sorted in every unit:", all(np.all(np.diff(u["spikes"]) >= 0) for s in sessions for u in s["units"]))
print("trial_time_index increasing in every file:", all(np.all(np.diff(s["trials"].trial_time_index) > 0) for s in sessions))
tti = np.array([s["trials"].trial_time_index.max() / 60000 for s in sessions])
print(f"last trial_time_index per session, read as ms: {tti.min():.1f} to {tti.max():.1f} min")

# Trial durations and beep times against the Methods (fixation 1-5 s; beep after 1-1.5 s of fixation)
rows = []
for s in sessions:
    t = s["trials"]
    d = (t.stop_time - t.start_time).values * 1000
    b = t.beep_time_ms.values
    nb = np.isfinite(b)
    rows.append(dict(monkey=s["monkey"], dur_fix=d[~nb], dur_beep=d[nb], beep=b[nb], post=d[nb] - b[nb]))
for m in L.MONKEYS:
    r = [x for x in rows if x["monkey"] == m]
    dfx = np.concatenate([x["dur_fix"] for x in r]); bb = np.concatenate([x["beep"] for x in r])
    post = np.concatenate([x["post"] for x in r])
    print(f"{L.SHORT[m]}: fixation-only trial duration {np.percentile(dfx, [0, 5, 50, 95, 100]).round(0)} ms, "
          f"{np.mean(dfx < 1000):.1%} under 1 s")
    print(f"    beep time from trial start {np.percentile(bb, [0, 5, 50, 95, 100]).round(0)} ms; "
          f"{np.mean((bb >= 1000) & (bb <= 1500)):.1%} within 1-1.5 s; {np.sum(bb < 0)} before the start; "
          f"{np.sum(post < 0)} after the trial's end; time from beep to end, median {np.median(post):.0f} ms")

pp = L.pupil_preprocessing_check(sessions)
print("\nPupil, per session over all stored samples:")
print(pp[["mean", "sd", "mean_trace_absmax", "boxcar_change"]].describe().loc[["min", "50%", "max"]].round(3))
nan_in = np.mean([np.isnan(s["pupil"][i, :int(round((s["trials"].stop_time[i] - s["trials"].start_time[i]) * 1000))]).mean()
                  for s in sessions for i in range(len(s["trials"]))])
print(f"NaN samples inside trials: {nan_in:.2%}")

# Duplicates: identical spike trains or identical pupil rows across files
import hashlib
hs, hp = {}, {}
for s in sessions:
    for u in s["units"]:
        hs.setdefault(hashlib.md5(np.round(u["spikes"], 4).tobytes()).hexdigest(), []).append((s["session"], u["source_unit_index"]))
    for i, row in enumerate(s["pupil"]):
        x = row[:300]
        if np.isfinite(x).all():
            hp.setdefault(hashlib.md5(np.round(x, 5).tobytes()).hexdigest(), set()).add(s["session"])
print("identical spike trains across units:", [v for v in hs.values() if len(v) > 1])
print("pupil rows shared between sessions:", sum(len(v) > 1 for v in hp.values()))

The stored trace is consistent with the Methods' preprocessing having been applied before deposit:
over each session the samples have mean near 0 and SD near 1 (z-scored, then reduced a little by
the subtraction of the mean trace), the mean trace across trials stays near 0, and a further 151 ms
boxcar changes the trace by about 5% of its SD (a raw 1 kHz trace would change more). This is
consistent with, not proof of, the preprocessing. The analysis uses the stored trace as the code
does. The trial start is taken as fixation onset, as in the code; the files do not say what event
it is (see the check of spikes outside the trial below).

Clock check: per session, the LC rate 0 to 200 ms after the beep minus the 200 ms before it, and the
pupil 300 to 800 ms after the beep minus the 100 ms before it, at the true beep times and at times
600 ms earlier (inside the same trials). A response at the true times and none at the shifted ones
shows that spikes and pupil share the trial table's clock.

In [ ]:
clk = L.clock_check(sessions)
ok = clk.dropna(subset=["p_lc_true"])
print(f"sessions with at least 6 usable beep trials: {len(ok)} of {len(clk)}; "
      f"too few: {list(clk.session[clk.p_lc_true.isna()])}")
for m in ("Oz", "Ci"):
    k = ok[ok.monkey == m]
    print(f"{m}: LC response true {k.lc_true.median():.2f} sp/s (sessions p<0.05: {(k.p_lc_true < 0.05).sum()}/{len(k)}), "
          f"shifted {k.lc_shifted.median():.2f} ({(k.p_lc_shifted < 0.05).sum()}/{len(k)}); "
          f"pupil true {k.pupil_true.median():.3f} ({(k.p_pupil_true < 0.05).sum()}/{len(k)}), "
          f"shifted {k.pupil_shifted.median():.3f} ({(k.p_pupil_shifted < 0.05).sum()}/{len(k)})")
noresp = ok[(ok.p_lc_true >= 0.05) & (ok.p_pupil_true >= 0.05)]
print("sessions where neither LC nor pupil responds at the true times (clock not established):", list(noresp.session))

Spikes outside the trial: where each stored spike falls against its trial (between the start and
the end, before the start, after the end), the rate in the 200 ms before and after the end of the
trial, and the beep trials whose 200 ms evoked rate window runs past the end of the trial.


In [ ]:
n_tot = n_in = 0
pre, post = [], []
r_before, r_after = [], []
for s in sessions:
    t = s["trials"]
    st, en = t.start_time.values, t.stop_time.values
    for u in s["units"]:
        sp = u["spikes"]
        n_tot += sp.size
        k = np.searchsorted(st, sp, "right") - 1          # trial whose anchor precedes the spike
        kk = np.clip(k, 0, None)
        rel = np.where(k >= 0, sp - st[kk], -np.inf)
        dur = (en - st)[kk]
        inside = (rel >= 0) & (rel <= dur)
        after = (k >= 0) & (rel > dur) & (rel - dur < 50)  # within 50 s after the end of trial k
        before = ~inside & ~after                           # the rest lie before the next trial's anchor
        n_in += int(inside.sum())
        post.append((rel - dur)[after])
        nxt = np.clip(k + 1, 0, len(st) - 1)
        pre.append((sp - st[nxt])[before])
        a = np.searchsorted(sp, en - 0.2); b = np.searchsorted(sp, en); c = np.searchsorted(sp, en + 0.2)
        r_before.append((b - a) / 0.2); r_after.append((c - b) / 0.2)
pre, post = np.concatenate(pre) * 1000, np.concatenate(post) * 1000
print(f"spikes: {n_tot}; inside a trial {n_in / n_tot:.1%}, before a trial start {pre.size / n_tot:.1%}, "
      f"after a trial end {post.size / n_tot:.1%}")
print("before the start, ms from the start (0, 1, 50, 99, 100 percentiles):", np.percentile(pre, [0, 1, 50, 99, 100]).round(0))
print("after the end, ms from the end (0, 50, 99, 100 percentiles):", np.percentile(post, [0, 50, 99, 100]).round(0))
print(f"mean rate over units and trials, 200 ms before the end {np.mean(np.concatenate(r_before)):.2f} sp/s, "
      f"200 ms after it {np.mean(np.concatenate(r_after)):.2f} sp/s")
fin = [np.isfinite(row[int(round((s["trials"].stop_time[j] - s["trials"].start_time[j]) * 1000)) + 1:]).sum()
       for s in sessions for j, row in enumerate(s["pupil"])]
print("finite pupil samples after the end of a trial, over all trials:", int(np.sum(fin)))
PT = pd.concat([L.pupil_trial_measures(s).assign(monkey=s["monkey"]) for s in sessions])
PB = PT[np.isfinite(PT.beep_ms) & PT.correct]
for m in L.MONKEYS:
    for rule in ("methods_ok", "code_ok"):
        k = PB[(PB.monkey == m) & PB[rule].astype(bool)]
        sh = (k.dur_ms - k.beep_ms) < L.EVOKED_FR_MS
        print(f"{L.SHORT[m]} {rule[:-3]} rule: {len(k)} beep trials, {int(sh.sum())} with less than 200 ms "
              f"from the beep to the end of the trial ({int(((k.dur_ms - k.beep_ms) < 0).sum())} with the beep after the end)")


Spikes are stored outside the trials: about 40% lie before a trial's start, up to a few seconds
before it (median about 0.8 s), but few lie after its end, and the rate falls from about 4.4 sp/s
in the last 200 ms of a trial to under 1 sp/s in the 200 ms after it (numbers above). The spike
record therefore runs from before the trial start to about its end. The pupil record has no
samples after the end of the trial either. So on beep trials whose 200 ms evoked window runs past
the end of the trial (printed above for each rule), the evoked rate is biased low. Both the
Methods' rule and the code's keep such trials, so the main readings keep them as well;
leaving them out is run as the variant "200 ms after beep" for every analysis below. It bears
only on measures that use the evoked rate; for the pupil measures alone the trial set changes
but no measure is biased.

That spikes are stored before the trial start does not tell what event the start marks. With the
beep a median 2.2 s after the start and the Methods' beep after 1 to 1.5 s of fixation, the start
may come before fixation onset, but the files hold no fixation marker to settle this. Every window
is taken from the trial start, as in the code, which reproduces the paper's numbers.


## Trial tables under each reading

`build_tables` applies the trial rule, the windows, the z-score exclusion and the drift residuals
(helper docstring). The counts of trials kept and dropped are printed for each reading.

In [ ]:
READINGS = {
    "M-E1": ("methods", "E1"),
    "M-E2": ("methods", "E2"),
    "code": ("code", "C"),
    "code trials + E1": ("code", "E1"),
    "Methods trials + code pupil": ("methods", "C"),
}
T = {}
for name, (rule, pupil) in READINGS.items():
    B, F = L.build_tables(sessions, rule, pupil)
    T[name] = (B, F, L.figure2_coefficients(B, F))
    cand = B.groupby("unit_key").n_beep_candidates.first().sum()
    print(f"{name:28s} beep rows {len(B):5d} of {cand} candidates (unit x trial), "
          f"dropped by |z| >= 8: {int(B.dropped_z.sum())}, fixation rows {len(F)}, units {B.unit_key.nunique()}, "
          f"sessions {B.session.nunique()}")
for rule in ("methods", "code"):
    n_all = sum(int(np.isfinite(s["trials"].beep_time_ms).sum()) for s in sessions)
    pt = pd.concat([L.pupil_trial_measures(s) for s in sessions])
    pt = pt[np.isfinite(pt.beep_ms)]
    col = "methods_ok" if rule == "methods" else "code_ok"
    print(f"{rule} trial rule keeps {int(pt[col].astype(bool).sum())} of {n_all} beep trials")
# Variant: Methods' trials that also leave the whole 200 ms evoked rate window before the end of the
# trial (spikes are mostly not stored past the trial end; see the data check).
POST = {s["session"]: L.pupil_trial_measures(s).set_index("trial").eval("dur_ms - beep_ms") for s in sessions}
for name in ("M-E1", "M-E2"):
    B, F, _ = T[name]
    keep = np.array([POST[s][t] >= L.EVOKED_FR_MS for s, t in zip(B.session, B.trial)])
    B2, F2 = B[keep].copy(), F.copy()
    L.add_residuals(B2, F2)
    T[name + ", 200 ms after beep"] = (B2, F2, L.figure2_coefficients(B2, F2))
    print(f"{name + ', 200 ms after beep':28s} beep rows {keep.sum():5d} of {len(B)}")
B0 = T["M-E1"][0]
REMOVED = L.code_removed_units(B0)
REMOVED_SESSIONS = [k.split("|")[1] for k in REMOVED
                    if B0[B0.session == k.split("|")[1]].unit_key.nunique() == 1]
print("units the code drops as rows 43 and 65 of its stats table:", REMOVED)
print("sessions that leave with them:", REMOVED_SESSIONS)
for name in ("M-E1", "M-E2", "code"):
    B = T[name][0]
    print(name, "evoked pupil range", B.pe.min().round(2), "to", B.pe.max().round(2),
          "; baseline pupil", B.pb.min().round(2), "to", B.pb.max().round(2),
          "; baseline rate", B.sb.min().round(1), "to", B.sb.max().round(1), "sp/s; evoked rate",
          B.se.min().round(1), "to", B.se.max().round(1), "sp/s")

## include-list-link

> The dandiset's sessions and units are those of the paper's code (its include lists), and the code's
> pipeline, run here on the dandiset, gives the statistics the code printed for it.

This is a registry check added by this attempt. Records are matched by session name (the file's
`session_id` plus `.mat`) and unit (`source_unit_index` minus 1, the code's `unit`). The code's
pipeline is the `code` reading, with the code's removal of rows 43 and 65 where the code applies it.

In [ ]:
sess_list, unit_list = L.include_lists()
ds_sess = {(s["monkey"], s["session"] + ".mat") for s in sessions}
ds_units = {(s["monkey"], s["session"] + ".mat", u["source_unit_index"] - 1) for s in sessions for u in s["units"]}
ls = set(map(tuple, sess_list[["monkey", "session_file"]].values))
lu = {(a, b, int(c)) for a, b, c in unit_list[["monkey", "session_file", "unit"]].values}
print(f"include list: {len(ls)} sessions, {len(lu)} units; dandiset: {len(ds_sess)} sessions, {len(ds_units)} units")
print("listed sessions not in the dandiset:", sorted(ls - ds_sess), "; dandiset sessions not listed:", sorted(ds_sess - ls))
print("listed units not in the dandiset:", sorted(lu - ds_units), "; dandiset units not listed:", sorted(ds_units - lu))
print("unit_id values absent from the list's 1..107 numbering:", sorted(set(range(1, 108)) - set(unit_list.unit_id)))

In [ ]:
txt = L.code_stats_text()
Bc, Fc, Cc = T["code"]
mine = {}
mine["beep trials"] = (len(Bc), int(re.search(r"included (\d+) beep trials", txt).group(1)))
mine["fixation trials"] = (len(Fc), int(re.search(r"and (\d+) non-beep", txt).group(1)))
# Figure 2 tests as the code prints them (its row numbering in brackets); Ci rows 1, 2, 4, 5, 6 over 63 units,
# row 3 over the code's sessions
labels = {1: "Baseline LC activity vs baseline pupil diameter (partial Spearman; residuals)",
          2: "Baseline-subtracted evoked LC responses vs baseline-subtracted evoked pupil responses",
          3: "Baseline-subtracted evoked pupil responses vs baseline pupil diameter (residuals)",
          4: "Baseline-subtracted evoked LC responses vs baseline LC activity (residuals)",
          5: "Baseline LC activity vs baseline-subtracted evoked pupil responses",
          6: "Baseline pupil diameter vs baseline-subtracted evoked LC responses"}
pt_code = L.population_tests(Cc, (), REMOVED_SESSIONS)
cmp = []
for row, lab in labels.items():
    for m in ("Oz", "Ci"):
        g = re.search(re.escape(lab) + rf": {m} p = ([\d.e-]+) \(N = (\d+), median rho = ([-\d.]+)\)", txt)
        r = pt_code[(pt_code.row == row) & (pt_code.monkey == m)].iloc[0]
        cmp.append(dict(row=row, monkey=m, n=r.n, n_code=int(g.group(2)), median=r["median"],
                        median_code=float(g.group(3)), p=float(f"{r.p_T1:.3g}"), p_code=float(g.group(1))))
cmp = pd.DataFrame(cmp)
cmp["match"] = (cmp.n == cmp.n_code) & ((cmp["median"] - cmp.median_code).abs() <= 0.0015) & \
               ((cmp.p - cmp.p_code).abs() <= 0.1 * cmp.p_code)
print(cmp.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
rng = L.range_correlations(Bc, Cc)
rc = []
for _, r in rng.iterrows():
    key = ("Baseline pupil diameter vs evoked LC relationship as a function of evoked LC range" if r.analysis.startswith("row 6")
           else "Baseline LC activity vs evoked pupil relationship as a function of evoked pupil range")
    g = re.search(re.escape(key) + rf" \(Spearman rho = ([-\d.]+), p = ([\d.e-]+) for monkey {r.monkey}\)", txt)
    rc.append(dict(monkey=r.monkey, analysis=r.analysis, rho=round(r.rho, 2), rho_code=float(g.group(1)),
                   p=float(f"{r.p:.3g}"), p_code=float(g.group(2))))
rc = pd.DataFrame(rc)
rc["match"] = ((rc.rho - rc.rho_code).abs() <= 0.005) & ((rc.p - rc.p_code).abs() <= 0.1 * rc.p_code)
print(rc.to_string(index=False))
ll = L.llr_counts(Bc, REMOVED, REMOVED_SESSIONS)
code_ll = {}
for lab, name in (("evoked FR vs baseline FR", "lc"), ("evoked pupil vs baseline FR", "pupil_on_lc"),
                  ("evoked FR vs baseline pupil", "lc_on_pupil")):
    blk = txt.split(f"For {lab},")[1]
    for m in ("Oz", "Ci"):
        code_ll[(m, name)] = re.search(rf"{m}: (\d+)/(\d+)", blk).groups()
blk = txt.split("For evoked pupil vs baseline pupil")[1]
for m in ("Oz", "Ci"):
    code_ll[(m, "pupil")] = re.search(rf"{m}: (\d+)/(\d+)", blk).groups()
ll["code"] = [ "/".join(code_ll[(r.monkey, r.comparison)]) for r in ll.itertuples()]
ll["here"] = [f"{r.n_above}/{r.n}" for r in ll.itertuples()]
ll["match"] = ll.code == ll.here
print(ll[["monkey", "comparison", "here", "code", "match"]].to_string(index=False))
f3 = L.fig3_pooled(Bc, Fc, Cc, group="unit")
f3a = f3[f3.group == "all"]
print("Figure 3 pooled, code's grouping (by within-session unit index):")
print(f3a[["monkey", "column", "n", "rho", "p"]].round(4).to_string(index=False))
n_match = int(cmp.match.sum() + rc.match.sum() + ll.match.sum() + sum(a == b for a, b in mine.values()))
n_total = len(cmp) + len(rc) + len(ll) + len(mine)
print("trial counts here and in the code's output:", mine)
print(f"values matching the code's printed output: {n_match} of {n_total}")

**Result: reproduced.** Every session and unit on the code's include lists is in the dandiset and
every one in the dandiset is listed (83 sessions, 105 units). Run here on the dandiset, the code's
pipeline gives its printed trial counts, Figure 2 tests, range correlations and LLR counts (the
count of matches is printed above). The include list numbers its units 1 to 107 without 43 and
65, so the legacy removal of two units is already applied to the list; the code then removes rows
43 and 65 of its own 105-row table again, which drops the two units printed above (code_mismatch).
The Figure 3 values match the code's printed ones when firing rates are z-scored by within-session
unit index across all sessions, which is what `zscore_tables.m` does (it groups by `unit`, not
`unit_id`), in place of per unit as the Figure 3 legend says.

## data-units-sessions

> The analysis used 42 LC single units from 32 sessions in monkey Oz and 63 units from 51 sessions in
> monkey Ci, recorded with pupil diameter during passive fixation with occasional startling beeps.

Expected: Oz 42 units and 32 sessions; Ci 63 units and 51 sessions; tolerance 0.

In [ ]:
for name in ("M-E1", "code"):
    B = T[name][0]
    listed = B[[(m, s + ".mat", u) in lu for m, s, u in zip(B.monkey, B.session, B.unit)]]
    g = listed.groupby("monkey").agg(units=("unit_key", "nunique"), sessions=("session", "nunique"),
                                      beep_rows=("trial", "size"))
    per_unit = listed.groupby("unit_key").size()
    print(name, "\n", g, "\n beep trials per unit: min", per_unit.min(), "median", per_unit.median(), "max", per_unit.max())
print("removing the code's two units leaves:", T["code"][0][~T["code"][0].unit_key.isin(REMOVED)]
      .groupby("monkey").agg(units=("unit_key", "nunique"), sessions=("session", "nunique")).to_dict())

**Result: reproduced.** All four counts match: Oz 42 units in 32 sessions, Ci 63 units in 51
sessions, each with beep trials, spikes and pupil, under both trial rules. The code's double
removal would give 61 units in 50 sessions for Ci.

## Figure 2: rows 1 to 6

Per unit (rows 1, 2, 4, 5, 6) or per session (row 3) Spearman coefficients, and per monkey the
population test T1 (Wilcoxon signed rank against 0, the code's `signrank`), with T2 (rank sum
against as many zeros) as a variant. The table gives every reading; `n_sig_pos` and `n_sig_neg`
count units or sessions with p < 0.05 of each sign. The paper's p-values are beside ours.

In [ ]:
PAPER_P = {(1, "Oz"): 8.6e-4, (1, "Ci"): 9.85e-6, (2, "Oz"): 0.0135, (2, "Ci"): 0.0018,
           (3, "Oz"): 1.16e-6, (3, "Ci"): 5.02e-9, (4, "Oz"): 5.88e-7, (4, "Ci"): 1.04e-11,
           (5, "Oz"): 0.894, (5, "Ci"): 0.904, (6, "Oz"): 0.915, (6, "Ci"): 0.008}
POP = {}
for name, (B, F, C) in T.items():
    P = L.population_tests(C)
    P["paper_p"] = [PAPER_P[(r, m)] for r, m in zip(P.row, P.monkey)]
    P.insert(0, "reading", name)
    POP[name] = P
    POP[name + ", code's removal"] = L.population_tests(C, REMOVED, REMOVED_SESSIONS).assign(reading=name + ", code's removal")
FIG2 = pd.concat(POP.values(), ignore_index=True)
for row in range(1, 7):
    print(f"Row {row}: {L.ROWS[row]}")
    print(FIG2[FIG2.row == row][["reading", "monkey", "n", "median", "p_T1", "p_T2", "paper_p", "n_sig_pos", "n_sig_neg"]]
          .to_string(index=False, float_format=lambda v: f"{v:.3g}"))
    print()

In [ ]:
C = T["M-E1"][2]
fig, ax = plt.subplots(1, 6, figsize=(15, 3), sharey=True)
for k, a in enumerate(ax, start=1):
    for j, m in enumerate(L.MONKEYS):
        v = C[(C.row == k) & (C.monkey == m)]
        a.scatter(np.full(len(v), j) + np.linspace(-0.2, 0.2, len(v)), v.r, s=8,
                  c=np.where(v.p < 0.05, "k", "0.7"))
        a.hlines(v.r.median(), j - 0.3, j + 0.3, color="r")
    a.axhline(0, color="0.5", lw=0.5)
    a.set_xticks([0, 1], ["Oz", "Ci"])
    a.set_title(f"row {k}", fontsize=9)
ax[0].set_ylabel("Spearman coefficient (M-E1)")
plt.tight_layout()
print("plotted coefficients range from", round(C.r.min(), 3), "to", round(C.r.max(), 3), "; none outside the axes")

**Result: reproduced (row 1).** In both monkeys the median partial coefficient is above zero and T1
gives p < 0.05 (table above, Oz p = 5.5e-4, Ci p = 1.4e-5 under the Methods' trials, against the
paper's 8.6e-4 and 9.85e-6). Row 1 does not depend on the evoked pupil, and every variant gives the
same.

**Result: reproduced (row 2).** Under E1 and under E2 the median is above zero and T1 gives
p < 0.05 in both monkeys (Oz p = 0.011 and 0.020, Ci p = 0.0028 and 0.0026; paper 0.0135 and
0.0018). The code reading gives Oz 0.0135 and Ci 0.0015.

**Result: reproduced (row 3).** Under E1 and E2 the median per-session coefficient is negative and T1
gives p < 1e-5 in both monkeys, over the paper's 32 and 51 sessions and over the code's 50.

**Result: reproduced (row 4).** The median per-unit coefficient is negative and T1 gives p < 1e-5 in
both monkeys under every reading (Oz 1.2e-6, Ci 1.5e-11 under the Methods' trials).

**Result: reproduced (row 5).** T1 gives p >= 0.05 in both monkeys under E1 and E2 (Oz 0.80 and
0.99, Ci 0.94 and 0.95; paper 0.894 and 0.904), with medians within 0.025 of zero. T2 and the code's
evoked pupil also give p >= 0.05.

**Result: reproduced (row 6).** Oz gives p >= 0.05 and Ci p < 0.05 with T1 (Oz 0.97, Ci 0.034 under
the Methods' trials; code reading 0.915 and 0.0105; paper 0.915 and 0.008), the paper's pattern.
Ci's median is small and negative (-0.022). With the code's removal of two Ci units, a variant
here, Ci gives p = 0.064 under the Methods' trials and 0.019 under the code reading. The Ci result is therefore fragile: it rests on the paper's denominator of 63 units, which the
claims file takes, and two units fewer move T1 across 0.05. Leaving out the beep trials whose 200 ms
evoked window runs past the end of the trial (a variant, see the data check) gives Ci p = 0.029 with
63 units and 0.046 with 61.


## cross-epoch-no-reliable-relation

> Despite consistent within-epoch relationships between LC activity and pupil size, there are no
> reliable relationships between the two across epochs.

Decided from rows 1, 2, 5 and 6 above.

In [ ]:
def row_state(P, row):
    g = P[P.row == row].set_index("monkey")
    return {m: (round(g.loc[m, "median"], 3), float(f"{g.loc[m, 'p_T1']:.3g}")) for m in ("Oz", "Ci")}

for name in ("M-E1", "M-E2", "code"):
    P = POP[name]
    within = all(P[(P.row == r)].p_T1.max() < 0.05 and (P[P.row == r]["median"] > 0).all() for r in (1, 2))
    cross_same_sign = {r: bool((P[P.row == r].p_T1 < 0.05).all() and len(set(np.sign(P[P.row == r]["median"]))) == 1)
                       for r in (5, 6)}
    print(name, "| rows 1 and 2 hold in both monkeys:", within, "| cross-epoch row significant with one sign in both:",
          cross_same_sign, "| row 5", row_state(P, 5), "| row 6", row_state(P, 6))

**Result: reproduced.** Rows 1 and 2 hold in both monkeys, and neither cross-epoch row is
significant with the same sign in both monkeys, under E1, E2 and the code reading.

## range-baseline-pupil-evoked-lc and range-baseline-lc-evoked-pupil

> The correlation between baseline pupil and evoked LC responses does not depend on the range of
> evoked LC responses in either monkey (Oz rho = 0.02, p = 0.89; Ci rho = 0.07, p = 0.589).

> The correlation between baseline LC activity and evoked pupil responses does not depend on the
> range of evoked pupil responses in Ci (rho = 0.18, p = 0.16), but in Oz it becomes more negative
> as that range grows (rho = -0.33, p = 0.03). Expected Oz rho -0.33 +/- 0.033.

Across units, per monkey, the Spearman correlation of each unit's row 6 (row 5) coefficient with the
range of its evoked firing rates (evoked pupil) over beep trials.

In [ ]:
RNG = pd.concat([L.range_correlations(B, C).assign(reading=name) for name, (B, F, C) in T.items()]
                + [L.range_correlations(B[~B.unit_key.isin(REMOVED)], C[~C.unit_key.isin(REMOVED)])
                   .assign(reading=name + ", code's removal") for name, (B, F, C) in T.items() if name in ("M-E1", "M-E2", "code")],
                ignore_index=True)
print(RNG[RNG.analysis.str.startswith("row 6")][["reading", "monkey", "n", "rho", "p"]].to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** The row 6 coefficient is not correlated with the range of evoked LC
responses in either monkey (Oz rho = 0.04, p = 0.78; Ci rho = 0.07, p = 0.60 under the Methods'
trials; code reading 0.02, p = 0.89 and 0.05, p = 0.69).

In [ ]:
print(RNG[RNG.analysis.str.startswith("row 5")][["reading", "monkey", "n", "rho", "p"]].to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: not reproduced.** Under the Methods' evoked pupil, read as E1 or E2, Ci gives p >= 0.05 as
the paper says, but Oz does not show the negative correlation (rho = -0.04, p = 0.80 under E1;
rho = -0.13, p = 0.40 under E2). The code reading gives the paper's numbers (Oz rho = -0.33,
p = 0.032; Ci rho = 0.18, p = 0.16), but neither mixture does (Oz rho = -0.12, p = 0.45 with the
code's trials and E1; rho = -0.20, p = 0.21 with the Methods' trials and the code's evoked pupil):
the Oz result needs both the code's trial rule, which keeps beep trials up to the padded end of
the pupil record, and its 0 to 800 ms evoked pupil.
Leaving out the beep trials whose 200 ms evoked window runs past the end of the trial (variant)
gives Oz rho = 0.01, p = 0.94 (E1) and -0.10, p = 0.52 (E2), the same outcome.


## figs1-raw-baselines-similar

> Using raw baseline measures in place of drift residuals gives results similar to those of Figure 2.

The six Figure 2 coefficients with raw baselines; 12 row and monkey cells are compared with the
paper's Figure 2 pattern (rows 1 to 4 significant in the reported direction in both monkeys, row 5
not significant in either, row 6 not significant in Oz and significant in Ci).

In [ ]:
PATTERN = {1: ("+", "+"), 2: ("+", "+"), 3: ("-", "-"), 4: ("-", "-"), 5: ("ns", "ns"), 6: ("ns", "+-")}

def cell_matches(row, monkey, med, p):
    want = PATTERN[row][0 if monkey == "Oz" else 1]
    if want == "ns":
        return p >= 0.05
    if want == "+-":
        return p < 0.05
    return p < 0.05 and (med > 0 if want == "+" else med < 0)

S1 = []
for name, (B, F, C) in T.items():
    P = L.population_tests(L.figure2_coefficients(B, F, baseline="raw"))
    P["match"] = [cell_matches(r.row, r.monkey, r["median"], r.p_T1) for _, r in P.iterrows()]
    P["match_T2"] = [cell_matches(r.row, r.monkey, r["median"], r.p_T2) for _, r in P.iterrows()]
    P.insert(0, "reading", name)
    S1.append(P)
S1 = pd.concat(S1, ignore_index=True)
print(S1[["reading", "row", "monkey", "n", "median", "p_T1", "p_T2", "match", "match_T2"]].to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print(S1.groupby("reading", sort=False)[["match", "match_T2"]].sum())

**Result: partially reproduced.** With raw baselines, 11 of the 12 cells match the paper's Figure 2
pattern under E1 and under E2; the cell that differs is row 6 in Ci (median -0.006, p = 0.14), which
is significant with residuals but not with raw baselines. The code reading matches all 12.
The variant without beep trials whose 200 ms evoked window runs past the end of the trial also
matches 11 of 12, with the same cell (row 6 in Ci, p = 0.13) differing.


## figs2-llr-pupil-baseline-evoked, figs2-llr-lc-baseline-evoked, figs2-llr-cross-epoch

Linear against quadratic fits by maximum likelihood; LLR = 2 x the difference in log-likelihood,
counted above 3.841. Per session for evoked pupil on baseline pupil, per unit otherwise. Expected:
pupil Oz 7/32, Ci 6/51; LC Oz 2/42, Ci 3/63; evoked pupil on baseline LC Oz 4/42, Ci 4/63; evoked LC
on baseline pupil Oz 2/42, Ci 5/63; tolerance one session or unit.

In [ ]:
EXPECTED_LLR = {("Oz", "pupil"): 7, ("Ci", "pupil"): 6, ("Oz", "lc"): 2, ("Ci", "lc"): 3,
                ("Oz", "pupil_on_lc"): 4, ("Ci", "pupil_on_lc"): 4, ("Oz", "lc_on_pupil"): 2, ("Ci", "lc_on_pupil"): 5}
LLR = []
for name, (B, F, C) in T.items():
    for rem_name, ru, rs in (("all", (), ()), ("code's removal", REMOVED, REMOVED_SESSIONS)):
        x = L.llr_counts(B, ru, rs)
        x.insert(0, "units", rem_name)
        x.insert(0, "reading", name)
        LLR.append(x)
LLR = pd.concat(LLR, ignore_index=True)
LLR["expected"] = [EXPECTED_LLR[(m, c)] for m, c in zip(LLR.monkey, LLR.comparison)]
LLR["within_1"] = (LLR.n_above - LLR.expected).abs() <= 1
LLR["fraction"] = LLR.n_above / LLR.n
print("fits with nothing to fit (too few trials or distinct values):",
      int((LLR.n_groups - LLR.n).sum()), "over all readings")
print(LLR[LLR.comparison == "pupil"].drop(columns=["comparison"]).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** Under E1 over the paper's sessions, 6 of 32 Oz sessions and 7 of 51 Ci
sessions exceed 3.841, both within one of the paper's 7 and 6 and both under half; under E2, 5/32
(outside the tolerance) and 5/51. The code reading gives exactly 7/32 and 6/51.

These counts depend on the trial set: leaving out the beep trials whose 200 ms evoked rate window
runs past the end of the trial (a variant aimed at the evoked rate, which this comparison does not
use) gives 3/32 and 9/51 under E1 and 2/32 and 5/51 under E2, outside the tolerance in at least one
monkey. The deciding reading keeps those trials, as the Methods' rule and the code do.


In [ ]:
print(LLR[LLR.comparison == "lc"].drop(columns=["comparison"]).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** Under the Methods' trials, 3 of 42 Oz units and 4 of 63 Ci units exceed
3.841, each within one of the paper's 2 and 3 (4 of 61 with the code's removal). The code reading
gives 2/42 and 3/63.

In [ ]:
X = LLR[LLR.comparison.isin(["pupil_on_lc", "lc_on_pupil"])]
print(X.drop(columns=["max_llr"]).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
summ = X.groupby(["reading", "units"], sort=False).agg(all_within=("within_1", "all"), max_fraction=("fraction", "max"))
print(summ)
# How close the cross-epoch LLRs lie to the cutoff: units within 3.841 +/- 1, and the counts at 2.841 and 4.841
near = []
for name in ("M-E1", "M-E2", "code"):
    B = T[name][0]
    for comp in ("pupil_on_lc", "lc_on_pupil"):
        x, y = L.LLR_PAIRS[comp]
        for m in L.MONKEYS:
            v = np.array([L.llr_quadratic(b[x], b[y]) for _, b in B[B.monkey == m].groupby("unit_key")])
            near.append(dict(reading=name, comparison=comp, monkey=L.SHORT[m], n=len(v),
                             n_above=int((v > L.LLR_CRIT).sum()), expected=EXPECTED_LLR[(L.SHORT[m], comp)],
                             n_above_cut_minus_1=int((v > L.LLR_CRIT - 1).sum()), n_above_cut_plus_1=int((v > L.LLR_CRIT + 1).sum()),
                             llr_within_1_of_cut=" ".join(f"{a:.2f}" for a in np.sort(v[np.abs(v - L.LLR_CRIT) < 1]))))
print(pd.DataFrame(near).to_string(index=False))
print("units expected above 3.841 by chance at 5%: Oz", round(0.05 * 42, 1), "Ci", round(0.05 * 63, 1))


**Result: not reproduced.** Under no reading the criterion names (E1 or E2, all units or the code's
removal) are all four counts within one of the paper's: evoked pupil on baseline LC gives 5/42 (Oz)
and 6/63 (Ci) under E1 and 6/42 and 7/63 under E2, against 4/42 and 4/63, and the Oz count is above
a tenth of the units under both, so the partial rule does not apply either. Evoked LC on baseline
pupil matches (Oz 2/42, Ci 6/63 under both). The code reading gives exactly the paper's 4/42,
4/63, 2/42 and 5/63 (4/61 and 5/61 with its removal). The counts stay small (at most 7 of 63 and
6 of 42 units), so the statement that a linear fit is favoured holds; the claim's counts do not,
under the Methods' evoked pupil.

The counts are close to the cutoff (second table, every unit whose LLR lies within one of 3.841).
Under E1, a cutoff one higher would give Oz 4 and Ci 3 for evoked pupil on baseline LC, and one lower
9 and 9, so the counts move by one or more with small changes in the fits. With the beep trials
whose 200 ms evoked window runs past the end of the trial left out (variant), the evoked pupil on
baseline LC counts are 6/42 and 5/63 under E1 and 8/42 and 7/63 under E2, so the outcome is the same.
The partial rule's bound (fewer than a tenth of the units) is strict next to the 5% false positive
rate of the cutoff, which alone would give about 2.1 of 42 and 3.2 of 63 units; the bound is the
claims file's and is applied as written.


## fig3-pooled-within-epoch and fig3-pooled-cross-epoch-null

Firing rates z-scored per unit over its baseline, raw evoked and fixation-only rates; pupil as
stored, which the data check shows to be z-scored per session already, so the main reading and the
code's coincide for pupil. Residuals recomputed; Spearman over pooled trials per monkey. Columns:

In [ ]:
print(L.FIG3_COLUMNS)
SIGN = {1: 1, 2: 1, 3: -1, 4: -1}
F3 = []
for name, (B, F, C) in T.items():
    for grp in ("unit_key", "unit"):
        x = L.fig3_pooled(B, F, C, group=grp)
        x.insert(0, "zscore", "per unit" if grp == "unit_key" else "code (within-session index)")
        x.insert(0, "reading", name)
        F3.append(x)
F3 = pd.concat(F3, ignore_index=True)
W = F3[(F3.group == "all") & F3.column.isin([1, 2, 3, 4])].copy()
W["match"] = [(p < 0.05) and np.sign(r) == SIGN[c] for p, r, c in zip(W.p, W.rho, W.column)]
print(W[["reading", "zscore", "monkey", "column", "n", "rho", "p", "match"]].to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print(W.groupby(["reading", "zscore"], sort=False).match.sum())

In [ ]:
# Variant: raw baselines in place of residuals (the code's plot_S1_combined.m), per-unit z-score
for name in ("M-E1", "M-E2"):
    B, F, C = T[name]
    Bz, Fz = L.zscore_rates(B, F, "unit_key")
    for m in L.MONKEYS:
        b, f = Bz[Bz.monkey == m], Fz[Fz.monkey == m]
        s = L.session_rows(b)
        vals = [L.spearman(np.r_[b.sb, f.sb], np.r_[b.pb, f.pb]), L.spearman(b.se, b.pe), L.spearman(s.pb, s.pe), L.spearman(b.sb, b.se)]
        print(name, "raw baselines", L.SHORT[m], [(round(r, 3), float(f"{p:.2g}")) for r, p in vals])

**Result: reproduced.** All four within-epoch columns give p < 0.05 with the sign of the
corresponding Figure 2 row in both monkeys, under E1 and E2 (8 of 8 cells each), and under every
variant reading and the code's z-scoring. With raw baselines in place of residuals (the code's
plot_S1_combined.m, a variant), column 1 is not significant in Ci (rho = 0.005, p = 0.6).

In [ ]:
Xn = F3[F3.column.isin([5, 6])].copy()
print(Xn[["reading", "zscore", "monkey", "group", "n_units", "column", "n", "rho", "p"]].to_string(index=False, float_format=lambda v: f"{v:.3g}"))
both = []
for (rd, z, g, c), d in Xn.groupby(["reading", "zscore", "group", "column"], sort=False):
    both.append(dict(reading=rd, zscore=z, group=g, column=c,
                     sig_same_sign_both=bool((d.p < 0.05).all() and len(set(np.sign(d.rho))) == 1),
                     sig_in=",".join(d.monkey[d.p < 0.05])))
both = pd.DataFrame(both)
print(both.to_string(index=False))
print("cells significant with one sign in both monkeys:", int(both.sig_same_sign_both.sum()), "of", len(both))

In [ ]:
# The code's legacy rows 43 and 65, and whether either unit is in a row 1 group under each reading
for name in ("M-E1", "M-E2", "code"):
    B, F, C = T[name]
    k = L.unit_order(B)
    r1 = C[C.row == 1].set_index("unit_key")
    x = k[k.legacy_row.isin([43, 65])].copy()
    x["r_row1"] = r1.loc[x.unit_key, "r"].values
    x["p_row1"] = r1.loc[x.unit_key, "p"].values
    x["group"] = np.where(x.p_row1 < 0.05, np.where(x.r_row1 > 0, "positive", "negative"), "none")
    print(name)
    print(x[["legacy_row", "monkey", "session", "unit", "r_row1", "p_row1", "group"]].to_string(index=False, float_format=lambda v: f"{v:.3g}"))
# Groups formed after the code's removal of the two units (as the code forms them)
G = []
for name in ("M-E1", "M-E2", "code"):
    B, F, C = T[name]
    x = L.fig3_pooled(B[~B.unit_key.isin(REMOVED)], F[~F.unit_key.isin(REMOVED)], C[~C.unit_key.isin(REMOVED)])
    G.append(x[x.group != "all"].assign(reading=name + ", code's removal"))
G = pd.concat(G, ignore_index=True)
print(G[["reading", "monkey", "group", "n_units", "column", "n", "rho", "p"]].to_string(index=False, float_format=lambda v: f"{v:.3g}"))
# Variant: pupil z-scored again per session over all stored samples before the measures are pooled
ZP = {s["session"]: (np.nanmean(s["pupil"]), np.nanstd(s["pupil"])) for s in sessions}
Z2 = []
for name in ("M-E1", "M-E2"):
    B, F, C = T[name]
    B, F = B.copy(), F.copy()
    for D in (B, F):
        D["pb"] = (D.pb - D.session.map(lambda s: ZP[s][0])) / D.session.map(lambda s: ZP[s][1])
    B["pe"] = B.pe / B.session.map(lambda s: ZP[s][1])
    L.add_residuals(B, F)
    Z2.append(L.fig3_pooled(B, F, C).assign(reading=name + ", pupil z-scored again"))
Z2 = pd.concat(Z2, ignore_index=True)
print(Z2[["reading", "monkey", "group", "column", "rho", "p"]].to_string(index=False, float_format=lambda v: f"{v:.3g}"))
for d, lab in ((G, "groups after the code's removal"), (Z2, "pupil z-scored again")):
    d = d[d.column.isin([5, 6])]
    nb_ = sum(bool((g.p < 0.05).all() and len(set(np.sign(g.rho))) == 1)
              for _, g in d.groupby(["reading", "group", "column"]))
    print(lab, "| cells significant with one sign in both monkeys:", nb_)


**Result: reproduced.** No cross-epoch column, in all units or in either group, is significant with
the same sign in both monkeys, under any reading. Ci's negative group (4 units) shows a significant
negative relation of evoked LC with baseline pupil (rho about -0.17, p about 0.01), not found in
Oz's 3 units.

Neither unit the code removes as rows 43 and 65 is in a row 1 group under any reading (their row 1
p is 0.15 to 0.42), so the groups are the same whether they are formed from all 63 Ci units or
from the code's 61, and the group results above are unchanged. Z-scoring the stored pupil again per
session (a variant) changes the pooled coefficients by at most about 0.01 and no conclusion.


## Summary

| Claim | Outcome |
|---|---|
| include-list-link (link) | reproduced |
| data-units-sessions | reproduced |
| fig2-row1-baseline-lc-pupil | reproduced |
| fig2-row2-evoked-lc-pupil | reproduced |
| fig2-row3-pupil-evoked-vs-baseline | reproduced |
| fig2-row4-lc-evoked-vs-baseline | reproduced |
| fig2-row5-baseline-lc-vs-evoked-pupil | reproduced |
| fig2-row6-baseline-pupil-vs-evoked-lc | reproduced |
| cross-epoch-no-reliable-relation | reproduced |
| range-baseline-pupil-evoked-lc | reproduced |
| range-baseline-lc-evoked-pupil | not reproduced |
| figs1-raw-baselines-similar | partially reproduced |
| figs2-llr-pupil-baseline-evoked | reproduced |
| figs2-llr-lc-baseline-evoked | reproduced |
| figs2-llr-cross-epoch | not reproduced |
| fig3-pooled-within-epoch | reproduced |
| fig3-pooled-cross-epoch-null | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The deciding readings follow the Methods where they
and the code differ (trial rule, the fixation-only window for the baseline rate, and the evoked
pupil window), as the claims file directs; the code's pipeline is run as a variant and reproduces
the code's printed output. The three claims that do not reproduce in full all turn on the evoked
pupil: measured from the beep to 20 ms before the end of fixation (Methods) in place of 0 to 800 ms
after the beep (code). The trial start is taken as fixation onset, as in the code.

**What was found about the dataset.** It holds exactly the paper's 83 sessions and 105 units, the
same as the code's include lists, with no duplicated recordings. The pupil trace appears to be
stored already preprocessed (z-scored, mean trace removed, smoothed) without saying so; its unit is
"a.u.". Spike times are stored against artificial trial anchors 100 s apart, with about 40% of them in the
few seconds before a trial's start and almost none after its end, and the real time of each
trial is only in `trial_time_index`, whose unit is not stated (milliseconds by its range). What event
the trial start marks is not stated, and beep times run from about 1 to 2.8 s after it, against
the Methods' 1 to 1.5 s of fixation; a few beeps fall before the trial start or after its end, and
some fixation-only trials last under 1 s against the Methods' 1 to 5 s. The session
`Cicero_022714_d18_402_FixnStim` is named for stimulation but has no stimulation marker. The
code's z-scoring for Figure 3 groups units by within-session index, and it removes two units twice.